In [1]:
# =========================
# Imports
# =========================

import requests
import pandas as pd

from pathlib import Path

import datetime as dt

In [2]:
# =========================
# Container Bronze
# =========================

try:

    # ------------------------------------------
    # Widget Databricks
    # Permite criar Jobs e receber parâmetros
    # ------------------------------------------

    dbutils.widgets.text(
        "BRONZE_CONTAINER",
        "bronze"
    )

    BRONZE_CONTAINER = (
        dbutils.widgets.get(
            "BRONZE_CONTAINER"
        ) or "bronze"
    )

    print(
        "✅ Widget Databricks carregado."
    )

except NameError:

    # ------------------------------------------
    # Execução Local
    # VSCode / Jupyter Notebook
    # ------------------------------------------

    print(
        "⚠️ dbutils não encontrado. "
        "Utilizando configuração local."
    )

    BRONZE_CONTAINER = "bronze"

# =========================
# Validação
# =========================

print(
    f"✅ Container Bronze: "
    f"{BRONZE_CONTAINER}"
)

⚠️ dbutils não encontrado. Utilizando configuração local.
✅ Container Bronze: bronze


In [3]:
# =========================
# Azure Blob Storage
# =========================

from azure.storage.blob import BlobServiceClient
import os

# Variáveis globais
AZURE_STORAGE_ACCOUNT = None
AZURE_STORAGE_KEY = None

try:

    # ------------------------------------------
    # Databricks + Azure Key Vault
    # ------------------------------------------

    AZURE_STORAGE_ACCOUNT = dbutils.secrets.get(
        scope="kvfiaptechprod",
        key="AZURE-STORAGE-ACCOUNT"
    )

    AZURE_STORAGE_KEY = dbutils.secrets.get(
        scope="kvfiaptechprod",
        key="AZURE-STORAGE-KEY"
    )

    print(
        "✅ Credenciais Azure Storage carregadas do Key Vault."
    )

except NameError:

    # ------------------------------------------
    # VSCode / Jupyter
    # ------------------------------------------

    print(
        "⚠️ dbutils não encontrado. "
        "Utilizando variáveis locais."
    )

    AZURE_STORAGE_ACCOUNT = os.getenv(
        "AZURE_STORAGE_ACCOUNT"
    )

    AZURE_STORAGE_KEY = os.getenv(
        "AZURE_STORAGE_KEY"
    )

except Exception as e:

    raise RuntimeError(
        f"❌ Erro ao carregar segredos do Key Vault: {e}"
    )

# ------------------------------------------
# Validação das credenciais
# ------------------------------------------

if not AZURE_STORAGE_ACCOUNT:

    raise ValueError(
        "❌ AZURE_STORAGE_ACCOUNT não configurado."
    )

if not AZURE_STORAGE_KEY:

    raise ValueError(
        "❌ AZURE_STORAGE_KEY não configurado."
    )

print(
    f"✅ Storage Account: "
    f"{AZURE_STORAGE_ACCOUNT}"
)

print(
    f"✅ Storage Key carregada: "
    f"{bool(AZURE_STORAGE_KEY)}"
)

# ------------------------------------------
# Inicialização Blob Storage
# ------------------------------------------

try:

    blob_service_client = BlobServiceClient(
        account_url=(
            f"https://{AZURE_STORAGE_ACCOUNT}.blob.core.windows.net"
        ),
        credential=AZURE_STORAGE_KEY
    )

    print(
        "✅ Cliente Azure Blob inicializado."
    )

except Exception as e:

    raise RuntimeError(
        f"❌ Erro ao inicializar Blob Storage: {e}"
    )

# ------------------------------------------
# Teste de conectividade
# ------------------------------------------

try:

    container_client = (
        blob_service_client.get_container_client(
            BRONZE_CONTAINER
        )
    )

    next(
        iter(
            container_client.list_blobs()
        ),
        None
    )

    print(
        f"✅ Acesso validado ao container "
        f"'{BRONZE_CONTAINER}'."
    )

except Exception as e:

    raise RuntimeError(
        f"❌ Erro ao acessar container Bronze: {e}"
    )

⚠️ dbutils não encontrado. Utilizando variáveis locais.
✅ Storage Account: stfiapoin4ci2kb4w7c
✅ Storage Key carregada: True
✅ Cliente Azure Blob inicializado.
✅ Acesso validado ao container 'bronze'.


In [4]:
# =========================
# API IBGE - Municípios
# =========================

url = (
    "https://servicodados.ibge.gov.br/"
    "api/v1/localidades/municipios"
)

response = requests.get(
    url,
    timeout=120
)

response.raise_for_status()

data = response.json()

print(
    f"✅ Municípios encontrados: "
    f"{len(data):,}"
)

✅ Municípios encontrados: 5,571


In [5]:
# =========================
# Normalização dos dados
# =========================

df = pd.json_normalize(data)

df = df.rename(
    columns={
        "id": "municipio_id",
        "nome": "municipio_nome",

        "microrregiao.id": "microrregiao_id",
        "microrregiao.nome": "microrregiao_nome",

        "microrregiao.mesorregiao.id": "mesorregiao_id",
        "microrregiao.mesorregiao.nome": "mesorregiao_nome",

        "microrregiao.mesorregiao.UF.id": "uf_id",
        "microrregiao.mesorregiao.UF.sigla": "uf_sigla",
        "microrregiao.mesorregiao.UF.nome": "uf_nome",

        "microrregiao.mesorregiao.UF.regiao.id": "regiao_id",
        "microrregiao.mesorregiao.UF.regiao.sigla": "regiao_sigla",
        "microrregiao.mesorregiao.UF.regiao.nome": "regiao_nome"
    }
)

# Auditoria

df["_ingested_at"] = (
    dt.datetime.now(
        dt.timezone.utc
    ).isoformat()
)

print(
    f"✅ Registros carregados: "
    f"{len(df):,}"
)

df.head()

✅ Registros carregados: 5,571


,municipio_id,municipio_nome,microrregiao_id,microrregiao_nome,mesorregiao_id,mesorregiao_nome,uf_id,uf_sigla,uf_nome,regiao_id,...,regiao-imediata.regiao-intermediaria.id,regiao-imediata.regiao-intermediaria.nome,regiao-imediata.regiao-intermediaria.UF.id,regiao-imediata.regiao-intermediaria.UF.sigla,regiao-imediata.regiao-intermediaria.UF.nome,regiao-imediata.regiao-intermediaria.UF.regiao.id,regiao-imediata.regiao-intermediaria.UF.regiao.sigla,regiao-imediata.regiao-intermediaria.UF.regiao.nome,microrregiao,_ingested_at
0,1100015,Alta Floresta D'Oeste,11006.0,Cacoal,1102.0,Leste Rondoniense,11.0,RO,Rondônia,1.0,...,1102,Ji-Paraná,11,RO,Rondônia,1,N,Norte,NaN,2026-08-23T23:44:19.303611+00:00
1,1100023,Ariquemes,11003.0,Ariquemes,1102.0,Leste Rondoniense,11.0,RO,Rondônia,1.0,...,1101,Porto Velho,11,RO,Rondônia,1,N,Norte,NaN,2026-08-23T23:44:19.303611+00:00
2,1100031,Cabixi,11008.0,Colorado do Oeste,1102.0,Leste Rondoniense,11.0,RO,Rondônia,1.0,...,1102,Ji-Paraná,11,RO,Rondônia,1,N,Norte,NaN,2026-08-23T23:44:19.303611+00:00
3,1100049,Cacoal,11006.0,Cacoal,1102.0,Leste Rondoniense,11.0,RO,Rondônia,1.0,...,1102,Ji-Paraná,11,RO,Rondônia,1,N,Norte,NaN,2026-08-23T23:44:19.303611+00:00
4,1100056,Cerejeiras,11008.0,Colorado do Oeste,1102.0,Leste Rondoniense,11.0,RO,Rondônia,1.0,...,1102,Ji-Paraná,11,RO,Rondônia,1,N,Norte,NaN,2026-08-23T23:44:19.303611+00:00


In [6]:
# =========================
# Salvar Parquet
# =========================

temp_dir = (
    Path.cwd() / "tmp"
)

temp_dir.mkdir(
    parents=True,
    exist_ok=True
)

date_suffix = dt.datetime.now().strftime(
    "%Y-%m-%d"
)

parquet_file = (
    temp_dir /
    f"{date_suffix}_ibge_municipios.parquet"
)

df.to_parquet(
    parquet_file,
    index=False
)

print(
    f"✅ Parquet criado: "
    f"{parquet_file}"
)

✅ Parquet criado: /home/linux/projetos/postech-aisc-fase-2-pipeline-azure-g23/jobs/tmp/2026-08-23_ibge_municipios.parquet


In [8]:
# =========================
# Upload Bronze
# =========================

blob_name = (
    #f"ibge/municipios/"
    f"{date_suffix}_ibge_municipios.parquet"
)

blob_client = (
    blob_service_client.get_blob_client(
        container=BRONZE_CONTAINER,
        blob=blob_name
    )
)

with open(
    parquet_file,
    "rb"
) as data:

    blob_client.upload_blob(
        data,
        overwrite=True
    )

print(
    f"✅ Upload concluído: "
    f"{blob_name}"
)

✅ Upload concluído: 2026-08-23_ibge_municipios.parquet
